<a href="https://colab.research.google.com/github/Lintangarif/Procurement-fraud-MAS/blob/main/hybrid_mas_procurement.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# ---------------------------------------------------------
# Environment Setup and Dependencies
# ---------------------------------------------------------

# Configure Kaggle API authentication
# Note: Replace '<YOUR_KAGGLE_API_TOKEN>' with your actual API token before execution.
os.environ['KAGGLE_API_TOKEN'] = "<YOUR_KAGGLE_API_TOKEN>"

# Install Explainable AI dependencies
!pip install shap -q

# Retrieve and extract the Procurement Invoice Fraud Dataset v1
!kaggle datasets download -d tokelomashile2/procurement-invoice-fraud-dataset
!unzip -o -q procurement-invoice-fraud-dataset.zip -d procurement_data
!rm procurement-invoice-fraud-dataset.zip

print("[INFO] Initialization complete: Dataset successfully extracted to local directory.")

In [ ]:
"""
Hybrid Procurement Fraud Detection System
A Multi-Agent Architecture integrating Ensemble Machine Learning,
Deterministic Rule-Based Logic, and SHAP Explainability.
Dataset: Procurement Invoice Fraud Dataset v1
"""

import warnings
warnings.filterwarnings('ignore')
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.preprocessing import LabelEncoder
import shap

np.random.seed(42)

# -----------------------------------------------------------------------------
# 1. DataAgent: Dataset Ingestion and Preprocessing
# -----------------------------------------------------------------------------
class DataAgent:
    def __init__(self, base_path="procurement_data", n_samples=5000):
        self.base_path = base_path
        self.n_samples = n_samples
        self.df = None
        self.feature_names = []
        self.encoders = {}

    def _find_file(self, filename):
        """Recursively locate files to handle varying extraction path structures."""
        search_path = os.path.join(self.base_path, "**", filename)
        files = glob.glob(search_path, recursive=True)
        if not files:
            raise FileNotFoundError(f"Error: {filename} missing in {self.base_path}")
        return files[0]

    def load_and_prepare(self):
        print("[DataAgent] Initializing dataset loading and merging sequence...")
        path_inv = self._find_file("invoices.parquet")
        path_lab = self._find_file("labels.parquet")
        path_sup = self._find_file("suppliers.parquet")

        df_inv = pd.read_parquet(path_inv)
        df_lab = pd.read_parquet(path_lab)
        df_sup = pd.read_parquet(path_sup)

        df_merged = pd.merge(df_inv, df_lab, on='invoice_id')
        df_final = pd.merge(df_merged, df_sup, on='supplier_id')

        # Stratified sampling constraint
        n = min(self.n_samples, len(df_final))
        self.df = df_final.sample(n=n, random_state=42).reset_index(drop=True)

        # Drop non-predictive and metadata columns
        cols_to_drop = ['image_path', 'fraud_type', 'fraud_tags', 'explanations',
                        'invoice_id', 'supplier_id', 'department_id', 'invoice_date']
        self.df = self.df.drop(columns=[c for c in cols_to_drop if c in self.df.columns])

        # Categorical feature encoding mapping
        for col in self.df.select_dtypes(include=['object']).columns:
            le = LabelEncoder()
            self.df[col] = le.fit_transform(self.df[col].astype(str))
            self.encoders[col] = le

        self.feature_names = [c for c in self.df.columns if c != 'is_fraud']
        n_fraud = self.df['is_fraud'].sum()
        print(f"[DataAgent] Preprocessing complete. Target space: {n} records ({n_fraud} anomalous, {n - n_fraud} legitimate).")
        return self.df

    def split(self, test_size=0.2):
        X = self.df.drop(columns=['is_fraud'])
        y = self.df['is_fraud']
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_size, stratify=y, random_state=42
        )
        print(f"[DataAgent] Stratified partitioning successful (Train: {len(X_train)}, Test: {len(X_test)}).")
        return X_train, X_test, y_train, y_test


# -----------------------------------------------------------------------------
# 2. RuleAgent: Deterministic Compliance Logic Validation
# -----------------------------------------------------------------------------
class RuleAgent:
    def apply(self, df):
        rules = pd.DataFrame(index=df.index)

        # Define the 6 explicit business heuristic constraints
        rules['R1_Blacklisted'] = (df['blacklisted_flag'] == 1).astype(int)
        rules['R2_High_Risk_Score'] = (df['supplier_risk_score'] > 0.8).astype(int)
        rules['R3_Amount_Spike'] = (df['invoice_amount'] > (df['avg_invoice_amount'] * 2)).astype(int)
        rules['R4_New_Sup_High_Val'] = ((df['supplier_age_days'] < 30) & (df['invoice_amount'] > df['avg_invoice_amount'])).astype(int)
        rules['R5_Odd_Hour'] = ((df['submission_hour'] < 6) | (df['submission_hour'] > 20)).astype(int)
        rules['R6_Round_Amount'] = ((df['invoice_amount'] % 1000 == 0) & (df['invoice_amount'] > 0)).astype(int)

        rules['rule_score'] = rules.sum(axis=1)
        rules['rules_triggered'] = rules.iloc[:, :6].apply(
            lambda row: ', '.join([c.split('_', 1)[1] for c in row.index if row[c] == 1]), axis=1
        )
        print(f"[RuleAgent] Execution complete. Mean rule violation per transaction: {rules['rule_score'].mean():.2f}")
        return rules


# -----------------------------------------------------------------------------
# 3. MLAgent: Probabilistic Ensemble Engine
# -----------------------------------------------------------------------------
class MLAgent:
    def __init__(self, rf_weight=0.5, gb_weight=0.5):
        self.rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
        self.gb = GradientBoostingClassifier(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42)
        self.rf_weight = rf_weight
        self.gb_weight = gb_weight

    def train(self, X_train, y_train):
        self.rf.fit(X_train, y_train)
        self.gb.fit(X_train, y_train)
        print(f"[MLAgent] Training phase converged. Aggregation weights set to RF: {self.rf_weight} | GB: {self.gb_weight}")

    def predict(self, X):
        rf_prob = self.rf.predict_proba(X)[:, 1]
        gb_prob = self.gb.predict_proba(X)[:, 1]
        ensemble_prob = self.rf_weight * rf_prob + self.gb_weight * gb_prob
        return ensemble_prob, rf_prob, gb_prob


# -----------------------------------------------------------------------------
# 4. ExplainerAgent: Interpretability via SHAP Attribution
# -----------------------------------------------------------------------------
class ExplainerAgent:
    def __init__(self, model, X_train, feature_names):
        self.feature_names = feature_names
        self.explainer = shap.TreeExplainer(model)

    def explain_global(self, X_test):
        print("[ExplainerAgent] Initiating SHAP value extraction. Computing permutations...")
        shap_vals = self.explainer.shap_values(X_test)

        # Handle diverse SHAP output dimensions across library versions
        if isinstance(shap_vals, list):
            sv = shap_vals[1]
        elif len(np.shape(shap_vals)) == 3:
            sv = shap_vals[:, :, 1]
        else:
            sv = shap_vals

        plt.figure(figsize=(10, 6))
        shap.summary_plot(sv, X_test, feature_names=self.feature_names, show=False)
        plt.title("Global SHAP Summary — Feature Impact on Fraud Prediction")
        plt.tight_layout()
        plt.savefig('fig3_shap_global.png', dpi=300, bbox_inches='tight')
        plt.show()
        print("[ExplainerAgent] Global feature attribution mapping concluded.")
        return sv

    def explain_local(self, X_test, indices, sv):
        for idx in indices:
            pos = list(X_test.index).index(idx) if idx in X_test.index else idx
            vals = sv[pos]
            fig, ax = plt.subplots(figsize=(10, 4))
            colors = ['#ff4444' if v > 0 else '#4488ff' for v in vals]
            sorted_idx = np.argsort(np.abs(vals))[::-1]
            top_n = min(8, len(vals))

            ax.barh(range(top_n), vals[sorted_idx[:top_n]], color=[colors[i] for i in sorted_idx[:top_n]])
            ax.set_yticks(range(top_n))
            ax.set_yticklabels([self.feature_names[i] for i in sorted_idx[:top_n]])
            ax.set_xlabel('SHAP Value (impact on fraud probability)')
            ax.set_title(f'Local Explanation — Transaction #{idx}')
            ax.invert_yaxis()
            plt.tight_layout()
            plt.savefig(f'fig4_shap_local_{idx}.png', dpi=300, bbox_inches='tight')
            plt.show()


# -----------------------------------------------------------------------------
# 5. OrchestratorAgent: Multi-Agent Pipeline Coordination
# -----------------------------------------------------------------------------
class OrchestratorAgent:
    def __init__(self):
        self.data_agent = DataAgent(n_samples=5000)
        self.rule_agent = RuleAgent()
        self.ml_agent = MLAgent(rf_weight=0.5, gb_weight=0.5)

    def run(self):
        print("=" * 70)
        print("  HYBRID PROCUREMENT FRAUD DETECTION SYSTEM")
        print("  Multi-Agent Architecture Integration Testing")
        print("=" * 70 + "\n")

        # Phase 1: Data Ingestion
        self.data_agent.load_and_prepare()
        X_train, X_test, y_train, y_test = self.data_agent.split()

        # Phase 2: Deterministic Rule Application
        test_df = self.data_agent.df.loc[X_test.index]
        rules_test = self.rule_agent.apply(test_df)

        # Phase 3: Probabilistic Modeling
        self.ml_agent.train(X_train, y_train)
        ensemble_prob, rf_prob, gb_prob = self.ml_agent.predict(X_test)

        # Phase 4: Dynamic Decision Fusion Logic
        max_rules = 6
        rule_score_norm = rules_test['rule_score'].values / max_rules
        w_ml, w_rule = 0.75, 0.25
        base_hybrid = (w_ml * ensemble_prob) + (w_rule * rule_score_norm)

        # Implement conditional heuristic override based on regulatory constraints
        hybrid_score = np.where(ensemble_prob > 0.85, ensemble_prob, base_hybrid)
        hybrid_score = np.where(
            (ensemble_prob <= 0.85) & (rules_test['rule_score'].values >= 2),
            np.clip(hybrid_score + (0.3 * rule_score_norm), 0, 1),
            hybrid_score
        )

        optim_threshold = 0.30
        hybrid_pred = (hybrid_score >= optim_threshold).astype(int)

        # Phase 5: Metric Evaluation
        self._evaluate(y_test, hybrid_pred, hybrid_score, ensemble_prob, rf_prob, gb_prob)

        # Phase 6: XAI Generation
        explainer = ExplainerAgent(self.ml_agent.rf, X_train, self.data_agent.feature_names)
        sv = explainer.explain_global(X_test)
        high_risk_idx = X_test.index[np.argsort(hybrid_score)[-3:]]
        explainer.explain_local(X_test, high_risk_idx, sv)

        # Phase 7: Regulatory Audit Output
        self._audit_report(X_test, y_test, hybrid_score, hybrid_pred, rules_test, ensemble_prob)
        print("\n[OrchestratorAgent] Pipeline execution successfully terminated.\n")

    def _evaluate(self, y_test, hybrid_pred, hybrid_score, ensemble_prob, rf_prob, gb_prob):
        models = {
            'RF Standalone': ((rf_prob >= 0.5).astype(int), rf_prob),
            'GB Standalone': ((gb_prob >= 0.5).astype(int), gb_prob),
            'Ensemble ML Baseline': ((ensemble_prob >= 0.5).astype(int), ensemble_prob),
            'Proposed Hybrid MAS': (hybrid_pred, hybrid_score),
        }
        rows = []
        for name, (pred, prob) in models.items():
            rows.append({
                'Model Architecture': name,
                'Accuracy': accuracy_score(y_test, pred),
                'Precision': precision_score(y_test, pred, zero_division=0),
                'Recall': recall_score(y_test, pred, zero_division=0),
                'F1-Score': f1_score(y_test, pred, zero_division=0),
                'ROC-AUC': roc_auc_score(y_test, prob),
            })
        print("\n[Metrics] Comparative Architecture Performance:\n")
        print(pd.DataFrame(rows).set_index('Model Architecture').round(4).to_string())

        # Generate evaluation plots
        plt.figure(figsize=(8, 6))
        for name, (_, prob) in models.items():
            fpr, tpr, _ = roc_curve(y_test, prob)
            auc = roc_auc_score(y_test, prob)
            plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')
        plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC-AUC Trade-off Analysis')
        plt.legend()
        plt.tight_layout()
        plt.savefig('fig2_roc_curves.png', dpi=300, bbox_inches='tight')
        plt.show()

    def _audit_report(self, X_test, y_test, hybrid_score, hybrid_pred, rules_test, ml_prob):
        print("\n" + "-" * 70)
        print("  TRANSACTIONAL AUDIT REPORT (Top 10 High-Risk Instances)")
        print("-" * 70)

        report = X_test.copy()
        report['Ground_Truth'] = y_test.values
        report['ML_Confidence'] = ml_prob
        report['Rule_Violations'] = rules_test['rule_score'].values
        report['Triggered_Logic'] = rules_test['rules_triggered'].values
        report['Hybrid_Final_Score'] = hybrid_score
        report['Decision_Output'] = hybrid_pred

        report = report.sort_values('Hybrid_Final_Score', ascending=False)
        display_cols = ['invoice_amount', 'ML_Confidence', 'Rule_Violations',
                        'Triggered_Logic', 'Hybrid_Final_Score', 'Decision_Output', 'Ground_Truth']

        available_cols = [c for c in display_cols if c in report.columns]
        print(report[available_cols].head(10).to_string())

# -----------------------------------------------------------------------------
# System Execution Trigger
# -----------------------------------------------------------------------------
if __name__ == '__main__':
    orchestrator = OrchestratorAgent()
    orchestrator.run()

In [ ]:
from sklearn.model_selection import StratifiedKFold

# -----------------------------------------------------------------------------
# 6. Extended Evaluation: K-Fold Cross Validation (Table 4 mapping)
# -----------------------------------------------------------------------------
def run_kfold_evaluation(orchestrator, k=5):
    print("\n" + "-" * 70)
    print(f"  [Evaluation] {k}-Fold Cross Validation (Hybrid MAS)")
    print("-" * 70)

    # Retrieve full dataset from initialized DataAgent
    df = orchestrator.data_agent.df
    X = df.drop(columns=['is_fraud'])
    y = df['is_fraud']

    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    metrics = {'precision': [], 'recall': [], 'f1': []}

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Initialize and train MLAgent instance per fold
        ml_agent = MLAgent(rf_weight=0.5, gb_weight=0.5)

        # Suppress standard output during fold training to maintain log clarity
        import sys, os
        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        ml_agent.train(X_train, y_train)
        sys.stdout = old_stdout

        ensemble_prob, _, _ = ml_agent.predict(X_test)

        # Apply deterministic rules per fold
        rules_test = orchestrator.rule_agent.apply(X_test)

        # Replicate Hybrid Fusion Logic
        rule_score_norm = rules_test['rule_score'].values / 6.0
        base_hybrid = (0.75 * ensemble_prob) + (0.25 * rule_score_norm)

        hybrid_score = np.where(ensemble_prob > 0.85, ensemble_prob, base_hybrid)
        hybrid_score = np.where(
            (ensemble_prob <= 0.85) & (rules_test['rule_score'].values >= 2),
            np.clip(hybrid_score + (0.3 * rule_score_norm), 0, 1),
            hybrid_score
        )

        hybrid_pred = (hybrid_score >= 0.30).astype(int)

        # Compute fold metrics
        prec = precision_score(y_test, hybrid_pred, zero_division=0)
        rec = recall_score(y_test, hybrid_pred, zero_division=0)
        f1 = f1_score(y_test, hybrid_pred, zero_division=0)

        metrics['precision'].append(prec)
        metrics['recall'].append(rec)
        metrics['f1'].append(f1)

        print(f"[Fold {fold}] Precision: {prec:.4f} | Recall: {rec:.4f} | F1-Score: {f1:.4f}")

    print("-" * 70)
    print(f"Mean Precision : {np.mean(metrics['precision']):.4f} ± {np.std(metrics['precision']):.4f}")
    print(f"Mean Recall    : {np.mean(metrics['recall']):.4f} ± {np.std(metrics['recall']):.4f}")
    print(f"Mean F1-Score  : {np.mean(metrics['f1']):.4f} ± {np.std(metrics['f1']):.4f}")
    print("-" * 70)

# -----------------------------------------------------------------------------
# 7. Extended Evaluation: Ablation Study (Table 5 mapping)
# -----------------------------------------------------------------------------
def run_ablation_study(orchestrator):
    print("\n" + "-" * 70)
    print("  [Evaluation] Component Ablation & Sensitivity Analysis")
    print("-" * 70)

    # Utilize the terminal hold-out split
    X_train, X_test, y_train, y_test = orchestrator.data_agent.split()

    # Extract probability distributions
    ensemble_prob, _, _ = orchestrator.ml_agent.predict(X_test)
    rules_test = orchestrator.rule_agent.apply(X_test)
    rule_score_norm = rules_test['rule_score'].values / 6.0

    # Scenario 1: Standalone Ensemble (w_ml=1.0, threshold=0.5)
    pred_ml = (ensemble_prob >= 0.5).astype(int)

    # Scenario 2: Standalone Rule-Based (w_rules=1.0, threshold=0.30)
    pred_rule = (rule_score_norm >= 0.30).astype(int)

    # Scenario 3: Baseline Hybrid without heuristic boost (gamma=0.0)
    base_hybrid = (0.75 * ensemble_prob) + (0.25 * rule_score_norm)
    pred_hybrid_noboost = (base_hybrid >= 0.30).astype(int)

    # Scenario 4: Proposed Hybrid MAS with heuristic boost (gamma=0.30)
    hybrid_score = np.where(ensemble_prob > 0.85, ensemble_prob, base_hybrid)
    hybrid_score = np.where(
        (ensemble_prob <= 0.85) & (rules_test['rule_score'].values >= 2),
        np.clip(hybrid_score + (0.3 * rule_score_norm), 0, 1),
        hybrid_score
    )
    pred_hybrid_full = (hybrid_score >= 0.30).astype(int)

    scenarios = {
        "Pure ML (Standalone Ensemble)": pred_ml,
        "Pure Rule-Based": pred_rule,
        "Hybrid (No Boost)": pred_hybrid_noboost,
        "Proposed Hybrid MAS (Full)": pred_hybrid_full
    }

    for name, pred in scenarios.items():
        prec = precision_score(y_test, pred, zero_division=0)
        rec = recall_score(y_test, pred, zero_division=0)
        f1 = f1_score(y_test, pred, zero_division=0)
        print(f"{name: <30} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}")
    print("-" * 70 + "\n")

# -----------------------------------------------------------------------------
# Post-Pipeline Execution
# -----------------------------------------------------------------------------
# Assumes orchestrator object exists in memory from the main pipeline run
run_kfold_evaluation(orchestrator)
run_ablation_study(orchestrator)